## AIM 5014 Special Topics: AI Safety & Security
### Assignment 2: Mechanistic Interpretability

### **Due Friday, October 16th on Canvas**

In this assignment, we will explore how GPT-2 processes information using nnsight,a state-of-the-art library designed for tracing, editing, and patching model activations.  Review the course material from class 5 if you need a refresher on mechanistic interpretabiliy.

### Instructions

Run the cells in order. Then, answer questions 1 - 2 based on the output of the preceding cells. Enter your answers directly in the notebook.

In [ ]:
import sys
!{sys.executable} -m pip install -q --upgrade --only-binary=:all: "numpy<2.3" scikit-learn
!{sys.executable} -m pip install -q circuitsvis transformers nnsight

## Part 1: Attention

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from nnsight import LanguageModel

# Check if GPU is available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load the GPT-2 Small model using nnsight
# We explicitly request 'eager' attention implementation to allow attention weight retrieval
model = LanguageModel("gpt2", device_map=device, attn_implementation="eager")

# Set the configuration to output attentions
model.config.output_attentions = True

# ==========================================
# PART 1: Analyzing and Visualizing Attention Weights
# ==========================================
print("\n--- PART 1: Attention Weights Analysis ---")

prompt = "The Capital of France is Paris. The Capital of Germany is"

# Let's extract attention weights directly by targeting the attention module outputs
with model.trace(prompt) as tracer:
    # In the eager implementation, the attention module outputs a tuple of (attn_output, attn_weights)
    # We access the tuple components using .output and slice index [1] to fetch weights.
    attn_weights = model.transformer.h[5].attn.output[1].save()

# Get attention pattern for head 0. If it was already resolved directly, we read its value.
attention_tensor = attn_weights.value if hasattr(attn_weights, 'value') else attn_weights
attention_pattern = attention_tensor[0, 0].cpu().detach().numpy()

# Tokenize to get labels for the plot axes
tokens = model.tokenizer.tokenize(prompt)

# Plot the attention pattern matrix
fig, ax = plt.subplots(figsize=(8, 8))
im = ax.imshow(attention_pattern, cmap='viridis')

ax.set_xticks(np.arange(len(tokens)))
ax.set_yticks(np.arange(len(tokens)))
ax.set_xticklabels(tokens, rotation=45, ha="right")
ax.set_yticklabels(tokens)

plt.title("Attention Weights (nnsight): Layer 5 Head 0")
plt.colorbar(im)
plt.tight_layout()
plt.show()



### Question 1:

*   How does the resulting heatmap allow us to visualize the results of the attention mechanism?

*   In the attention weight plot for Layer 5 Head 0, look at the row representing the target query token "is" (either of the two occurrences). Which keys (prior tokens) are receiving the highest attention weights from "is"? Based on this, what role do you think this specific head is playing in routing contextual information?

* In the cell below change the example prompt given above ("The Capital of France is Paris. The Capital of Germany is") to one of your own and then run the heatmap code. Important: Make sure this is an example that will give you interesting results in an attention heatmap!  After you've run the code, interpret the results.

## Part 2: Activation Patching

In [ ]:

print("\n--- PART 2: Robust 2D Grid Activation Patching ---")

import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import torch

clean_prompt = "The capital of France is"
corrupted_prompt = "The capital of Italy is"

clean_tokens = model.tokenizer.tokenize(clean_prompt)
num_tokens = len(clean_tokens)
num_layers = len(model.transformer.h)

correct_token_id = model.tokenizer.encode(" Paris")[0]
incorrect_token_id = model.tokenizer.encode(" Rome")[0]

# 1. Gather all baseline clean AND corrupted activations cleanly
clean_activations = []
corrupted_activations = []

with model.trace(clean_prompt) as tracer:
    for layer_idx in range(num_layers):
        clean_activations.append(model.transformer.h[layer_idx].output[0].save())

with model.trace(corrupted_prompt) as tracer:
    for layer_idx in range(num_layers):
        corrupted_activations.append(model.transformer.h[layer_idx].output[0].save())
    corrupted_logits = model.lm_head.output[0, -1].save()

with model.trace(clean_prompt) as tracer:
    clean_logits = model.lm_head.output[0, -1].save()

def get_logit_diff(logits_val):
    return logits_val[correct_token_id] - logits_val[incorrect_token_id]

# Safely extract logits
clean_logits_val = clean_logits.value if hasattr(clean_logits, 'value') else clean_logits
corrupted_logits_val = corrupted_logits.value if hasattr(corrupted_logits, 'value') else corrupted_logits

clean_diff = get_logit_diff(clean_logits_val).item()
corrupted_diff = get_logit_diff(corrupted_logits_val).item()

# Initialize heatmap array
heatmap_results = np.zeros((num_layers, num_tokens))

# 2. Run the patching sweep
for layer_idx in range(num_layers):
    clean_val = clean_activations[layer_idx]
    if hasattr(clean_val, 'value'):
        clean_val = clean_val.value
    if clean_val.ndim == 3:
        clean_val = clean_val[0]

    corrupt_val = corrupted_activations[layer_idx]
    if hasattr(corrupt_val, 'value'):
        corrupt_val = corrupt_val.value
    if corrupt_val.ndim == 3:
        corrupt_val = corrupt_val[0]

    # Calculate the full difference tensor
    diff_tensor = clean_val - corrupt_val

    for token_idx in range(num_tokens):
        # Create a delta tensor of zeros, with the difference vector at our target token index
        patch_delta = torch.zeros_like(diff_tensor, device=device)
        patch_delta[token_idx] = diff_tensor[token_idx]

        with model.trace(corrupted_prompt) as tracer:
            # Apply the patch mathematically so nnsight tracks and propagates it dynamically
            if model.transformer.h[layer_idx].output[0].ndim == 3:
                model.transformer.h[layer_idx].output[0][0] = model.transformer.h[layer_idx].output[0][0] + patch_delta
            else:
                model.transformer.h[layer_idx].output[0] = model.transformer.h[layer_idx].output[0] + patch_delta

            patched_logits = model.lm_head.output[0, -1].save()

        patched_logits_val = patched_logits.value if hasattr(patched_logits, 'value') else patched_logits
        patched_diff = get_logit_diff(patched_logits_val).item()
        normalized_val = (patched_diff - corrupted_diff) / (clean_diff - corrupted_diff)
        heatmap_results[layer_idx, token_idx] = normalized_val

# Plot the heatmap
plt.figure(figsize=(10, 6))
sns.heatmap(
    heatmap_results,
    xticklabels=clean_tokens,
    yticklabels=list(range(num_layers)),
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0.0
)
plt.xlabel("Token Position Patched")
plt.ylabel("Layer Index")
plt.title("Residual Stream Activation Patching: Rome -> Paris")
plt.show()

### Question 2:

* How does activation patching work?

*  Look at the resulting activation patching heatmap. Which specific token position and layer(s) have the highest patching recovery values? What does this reveal about where and when the model retrieves the factual association that France's capital is Paris versus Italy's capital is Rome?